In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [1]:
import scipy.io as sio

# Load the meta.mat file
meta = sio.loadmat('data/imagenet/devkit/ILSVRC2012_devkit_t12/data/meta.mat', squeeze_me=True)['synsets']

# Filter out entries with no children (i.e., leaf nodes)
meta = [entry for entry in meta if entry[4] == 0]

# Create a mapping from class index to WNID
idx_to_wnid = {entry[0]: entry[1] for entry in meta}


import json

# Load the JSON file
with open('imagenet_class_index.json', 'r') as f:
    class_idx = json.load(f)

# Create a mapping from WNID to model class index
wnid_to_idx = {v[0]: int(k) for k, v in class_idx.items()}



# Read the ground truth labels
with open('data/imagenet/devkit/ILSVRC2012_devkit_t12/data/ILSVRC2012_validation_ground_truth.txt') as f:
    val_labels = [int(line.strip()) for line in f]

# Create the val_filenames.txt file with correct mappings
with open('data/imagenet/devkit/ILSVRC2012_devkit_t12/data/val_filenames.txt', 'w') as f:
    for i, label in enumerate(val_labels):
        fname = f"ILSVRC2012_val_{i+1:08d}.JPEG"
        wnid = idx_to_wnid[label]
        class_idx = wnid_to_idx[wnid]
        f.write(f"{fname} {class_idx}\n")



In [2]:
import os
from PIL import Image
from torch.utils.data import Dataset

class ImageNetValDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform=None):
        self.images_dir = images_dir
        self.transform = transform
        with open(labels_file, 'r') as f:
            lines = f.readlines()
        self.samples = [line.strip().split() for line in lines]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, label = self.samples[idx]
        image_path = os.path.join(self.images_dir, filename)
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(label)



In [3]:
def get_imagenet_loaders(batch_size, num_workers):
    data_dir = "data/imagenet"

    # -------------------------
    # Validation loader
    # -------------------------
    images_dir = os.path.join(data_dir, "val")
    labels_file = (
        "data/imagenet/devkit/ILSVRC2012_devkit_t12/data/val_filenames.txt"
    )

    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_dataset = ImageNetValDataset(
        images_dir=images_dir,
        labels_file=labels_file,
        transform=val_transform,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    # -------------------------
    # Training loader
    # -------------------------
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    train_dir = os.path.join(data_dir, "train")
    train_dataset = datasets.ImageFolder(
        root=train_dir,
        transform=train_transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )

    # -------------------------
    # Test loader
    # -------------------------
    # Standard ImageNet practice: reuse val as test
    test_loader = val_loader

    return train_loader, val_loader, test_loader


In [4]:
import torch
from torchvision import models, transforms
from torch.utils.data import DataLoader

# --------------------------------------------------
# Device
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Transforms (ImageNet standard)
# --------------------------------------------------
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# --------------------------------------------------
# Dataset + Loader (YOUR custom ImageNetValDataset)
# --------------------------------------------------
images_dir = "data/imagenet/val"
labels_file = "data/imagenet/devkit/ILSVRC2012_devkit_t12/data/val_filenames.txt"
batch_size = 64

dataset = ImageNetValDataset(
    images_dir=images_dir,
    labels_file=labels_file,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

ys = []
for _, y in dataloader:
    ys.append(y)
    break

y = ys[0]
print("Label min:", y.min().item())
print("Label max:", y.max().item())


# --------------------------------------------------
# Load pretrained ResNet-50
# --------------------------------------------------
model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V1
)
model = model.to(device)
model.eval()

# --------------------------------------------------
# Evaluation
# --------------------------------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

accuracy = 100.0 * correct / total
print(f"ResNet-50 ImageNet validation accuracy: {accuracy:.2f}%")


Label min: 23
Label max: 994
ResNet-50 ImageNet validation accuracy: 76.15%


In [5]:
# =========================
# Imports
# =========================
import torch
import torch.nn as nn
import torchvision.models as tv_models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Channel-wise gating (BN-safe)
# =========================
class ChannelGate(nn.Module):
    def __init__(self, channels, init_logit=5.0):
        super().__init__()
        # high positive init => gate ~ 0.993 (starts nearly open)
        self.logits = nn.Parameter(torch.ones(channels) * init_logit)

    def gate_values(self):
        return torch.sigmoid(self.logits)

    def forward(self, x):
        g = self.gate_values().view(1, -1, 1, 1)
        return x * g


# =========================
# Gate utilities
# =========================
@torch.no_grad()
def gather_all_gates(model: nn.Module) -> torch.Tensor:
    gates = []
    for m in model.modules():
        if isinstance(m, ChannelGate):
            gates.append(m.gate_values().detach().flatten())
    return torch.cat(gates) if len(gates) > 0 else torch.tensor([])


def gate_l1_sparsity_loss(model: nn.Module):
    """
    L1-style sparsity regularization on gate values.
    Use as:
        loss = ce_loss + beta * calib_loss + lambda_sparse * gate_l1_sparsity_loss(model)
    """
    vals = []
    for m in model.modules():
        if isinstance(m, ChannelGate):
            vals.append(m.gate_values())

    if len(vals) == 0:
        return torch.tensor(0.0, device=next(model.parameters()).device)

    return torch.cat(vals).mean()


@torch.no_grad()
def harden_gates(model: nn.Module, prune_ratio: float):
    """
    Global hardening per gate module:
      prune_ratio = 0.1  => prune 10%, keep 90% channels in each gate
    Sets logits to +/-10 so gates become ~binary.
    """
    for m in model.modules():
        if isinstance(m, ChannelGate):
            g = m.gate_values()
            n = g.numel()
            k = int(round((1.0 - prune_ratio) * n))

            if k <= 0:
                m.logits.fill_(-10.0)
                continue
            if k >= n:
                m.logits.fill_(10.0)
                continue

            thresh = torch.topk(g, k, largest=True).values.min()

            keep_mask = (g >= thresh).float()
            m.logits.copy_(torch.where(
                keep_mask > 0,
                torch.full_like(g, 10.0),
                torch.full_like(g, -10.0)
            ))


# =========================
# Gated Bottleneck
# =========================
class GatedBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)

        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn2   = nn.BatchNorm2d(planes)

        self.conv3 = nn.Conv2d(
            planes, planes * self.expansion, kernel_size=1, bias=False
        )
        self.bn3   = nn.BatchNorm2d(planes * self.expansion)

        # Gate AFTER bn3 (BN-safe)
        self.gate  = ChannelGate(planes * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        out = self.gate(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = self.relu(out)
        return out


# =========================
# Gated ResNet-50
# =========================
class GatedResNet50(nn.Module):
    def __init__(self, num_classes=1000, init_weights=False):
        super().__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(
            3, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64,  blocks=3, stride=1)
        self.layer2 = self._make_layer(128, blocks=4, stride=2)
        self.layer3 = self._make_layer(256, blocks=6, stride=2)
        self.layer4 = self._make_layer(512, blocks=3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * GatedBottleneck.expansion, num_classes)

        if init_weights:
            self._init_weights()

    def _make_layer(self, planes, blocks, stride):
        downsample = None
        out_channels = planes * GatedBottleneck.expansion

        if stride != 1 or self.in_planes != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_planes,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )

        layers = []
        layers.append(GatedBottleneck(self.in_planes, planes, stride, downsample))
        self.in_planes = out_channels

        for _ in range(1, blocks):
            layers.append(GatedBottleneck(self.in_planes, planes))

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


# =========================
# Load torchvision pretrained ResNet-50 weights into gated model
# =========================
def load_torchvision_pretrained_into_gated(model: GatedResNet50):
    """
    Copies weights from torchvision ResNet-50 into matching layers.
    Gate params remain as initialized.
    """
    try:
        from torchvision.models import resnet50, ResNet50_Weights
        base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    except Exception:
        # fallback for older torchvision
        base = tv_models.resnet50(pretrained=True)

    base_sd = base.state_dict()
    gated_sd = model.state_dict()

    copied = 0
    for k in gated_sd.keys():
        # skip gates entirely
        if "gate." in k:
            continue
        if k in base_sd and gated_sd[k].shape == base_sd[k].shape:
            gated_sd[k] = base_sd[k]
            copied += 1

    model.load_state_dict(gated_sd, strict=False)
    print(f"Loaded pretrained weights into gated model. Copied {copied} tensors.")
    return model


# =========================
# Build model
# =========================
model = GatedResNet50(num_classes=1000, init_weights=False)
model = load_torchvision_pretrained_into_gated(model)
model = model.to(DEVICE)

x = torch.randn(2, 3, 224, 224).to(DEVICE)
print(model(x).shape)  # torch.Size([2, 1000])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/ec2-user/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 65.1MB/s]

Loaded pretrained weights into gated model. Copied 320 tensors.
torch.Size([2, 1000])


In [6]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class SoftBinnedECELoss(nn.Module):
    """
    Soft-binned Expected Calibration Error (SB-ECE, L2 form)

    - Soft binning via Gaussian-like membership over confidence values
    - Bin anchors at:
        (1/(2m), 3/(2m), ..., (2m-1)/(2m))
    - Returns:
        sqrt( sum_b w_b * (conf_b - acc_b)^2 )

    Args:
        bins (int): number of confidence bins
        temperature (float): softness of bin membership
                             smaller => sharper assignment
        eps (float): numerical stability
    """
    def __init__(self, bins=15, temperature=0.05, eps=1e-8):
        super().__init__()
        self.bins = bins
        self.temperature = temperature
        self.eps = eps

        anchors = np.arange(1.0 / (2 * bins), 1.0, 1.0 / bins, dtype=np.float32)
        self.register_buffer("anchors", torch.tensor(anchors, dtype=torch.float32))

    def forward(self, logits, labels):
        """
        logits: [N, C]
        labels: [N]
        """
        probs = F.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)             # [N]
        acc = pred.eq(labels).float()             # [N]

        # [N, 1], [1, m]
        conf_tile = conf.unsqueeze(1)
        anchors = self.anchors.unsqueeze(0)

        # Soft membership (Gaussian-like kernel over anchors)
        # Shape: [N, m]
        scores = -((conf_tile - anchors) ** 2) / max(self.temperature, self.eps)
        c = F.softmax(scores, dim=1)

        # Per-bin soft counts
        sum_c = c.sum(dim=0)                                  # [m]
        sum_c_safe = torch.clamp(sum_c, min=self.eps)

        # Soft bin confidence / accuracy
        conf_b = (c * conf_tile).sum(dim=0) / sum_c_safe      # [m]
        acc_b  = (c * acc.unsqueeze(1)).sum(dim=0) / sum_c_safe  # [m]

        # Bin weights
        w = sum_c / torch.clamp(sum_c.sum(), min=self.eps)

        # L2 SB-ECE
        sb_ece = torch.sqrt(torch.sum(w * (conf_b - acc_b) ** 2) + self.eps)

        return sb_ece

In [7]:
@torch.no_grad()
def collect_logits_and_labels(model, loader, device):
    model.eval()
    logits_list, labels_list = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        logits_list.append(logits.detach().cpu())
        labels_list.append(y.detach().cpu())
    return torch.cat(logits_list, 0), torch.cat(labels_list, 0)

@torch.no_grad()
def acc_from_logits(logits, labels):
    return float((logits.argmax(1) == labels).float().mean().item())

@torch.no_grad()
def nll_from_logits(logits, labels):
    return float(F.cross_entropy(logits, labels, reduction="mean").item())

@torch.no_grad()
def ece_from_logits(logits, labels, n_bins=15, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    conf, pred = probs.max(1)
    acc = pred.eq(labels).float()

    edges = torch.linspace(0, 1, n_bins + 1, device=logits.device)
    ece = torch.zeros(1, device=logits.device)

    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        if b == n_bins - 1:
            inb = (conf >= lo) & (conf <= hi)
        else:
            inb = (conf >= lo) & (conf < hi)

        p = inb.float().mean()
        if p.item() > 0:
            ece += p * torch.abs(conf[inb].mean() - acc[inb].mean())

    return float(ece.item())

@torch.no_grad()
def ks_calibration_error(logits, labels, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=1)
    conf, pred = probs.max(1)
    correct = pred.eq(labels).float()

    conf_sorted, idx = torch.sort(conf)
    corr_sorted = correct[idx]
    denom = torch.arange(1, len(conf_sorted)+1, device=logits.device).float()
    cum_acc = torch.cumsum(corr_sorted, 0) / denom
    cum_conf = torch.cumsum(conf_sorted, 0) / denom
    return float(torch.max(torch.abs(cum_acc - cum_conf)).item())

@torch.no_grad()
def mean_sweep_ce(logits, labels, bin_list, temperature=1.0):
    vals = [ece_from_logits(logits, labels, n_bins=b, temperature=temperature) for b in bin_list]
    return float(np.mean(vals))

def fit_temperature(logits_val_cpu, labels_val_cpu, device, max_iter=50):
    logits = logits_val_cpu.to(device)
    labels = labels_val_cpu.to(device)

    logT = torch.zeros(1, device=device, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.5, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        T = torch.exp(logT)
        loss = F.cross_entropy(logits / T, labels)
        loss.backward()
        return loss

    opt.step(closure)
    T = float(torch.exp(logT).detach().cpu().item())
    return max(1e-3, min(T, 100.0))

@torch.no_grad()
def evaluate_all_metrics(model, val_loader, test_loader, device):
    logits_val, labels_val = collect_logits_and_labels(model, val_loader, device)
    logits_test, labels_test = collect_logits_and_labels(model, test_loader, device)

    logits_test = logits_test.to(device)
    labels_test = labels_test.to(device)

    # base
    acc = acc_from_logits(logits_test, labels_test)
    nll = nll_from_logits(logits_test, labels_test)
    ece = ece_from_logits(logits_test, labels_test, n_bins=ECE_BINS)
    ks  = ks_calibration_error(logits_test, labels_test)
    ms  = mean_sweep_ce(logits_test, labels_test, MEAN_SWEEP_BINS)

    # TS
    T = fit_temperature(logits_val, labels_val, device=device)
    ts_nll = nll_from_logits(logits_test / T, labels_test)
    ts_ece = ece_from_logits(logits_test, labels_test, n_bins=ECE_BINS, temperature=T)
    ts_ks  = ks_calibration_error(logits_test, labels_test, temperature=T)
    ts_ms  = mean_sweep_ce(logits_test, labels_test, MEAN_SWEEP_BINS, temperature=T)

    return {
        "acc": acc, "nll": nll, "ece": ece, "ts_ece": ts_ece,
        "ks": ks, "ts_ks": ts_ks,
        "mean_sweep_ce": ms, "ts_mean_sweep_ce": ts_ms,
        "ts_temperature": T,
        "ts_nll": ts_nll,
    }


In [21]:
import time

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_one_epoch_nll_sbece(
    model,
    loader,
    optimizer,
    device,
    nll_loss_fn,
    sbece_loss_fn,
    beta_sbece=1.0,       # keep consistent with CIFAR
    lambda_sparse=1e-4,
    sbece_every=1,
    grad_clip=None,
):
    model.train()

    total_samples = 0
    sum_total = 0.0
    sum_nll = 0.0
    sum_sbece = 0.0
    sum_sparse = 0.0
    correct = 0

    for step, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)

        # CE / NLL
        nll = nll_loss_fn(logits, y)

        # SB-ECE
        if sbece_every > 0 and (step % sbece_every) == 0:
            sbece = sbece_loss_fn(logits, y)
        else:
            sbece = logits.new_zeros(())

        # Sparsity
        sparse = gate_l1_sparsity_loss(model) if lambda_sparse > 0 else logits.new_zeros(())

        # Total
        loss = nll + beta_sbece * sbece + lambda_sparse * sparse
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        # Logging
        bs = y.size(0)
        total_samples += bs

        sum_total  += float(loss.item()) * bs
        sum_nll    += float(nll.item()) * bs
        sum_sbece  += float(sbece.item()) * bs
        sum_sparse += float(sparse.item()) * bs

        pred = logits.argmax(dim=1)
        correct += pred.eq(y).sum().item()

    denom = max(total_samples, 1)
    return {
        "loss":   sum_total / denom,
        "nll":    sum_nll / denom,
        "sbece":  sum_sbece / denom,
        "sparse": sum_sparse / denom,
        "acc1":   100.0 * correct / denom,
    }


@torch.no_grad()
def evaluate_one_epoch(
    model,
    loader,
    device,
    nll_loss_fn,
    sbece_loss_fn=None,
):
    model.eval()

    total_samples = 0
    sum_nll = 0.0
    sum_sbece = 0.0
    correct = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        nll = nll_loss_fn(logits, y)

        if sbece_loss_fn is not None:
            sbece = sbece_loss_fn(logits, y)
        else:
            sbece = logits.new_zeros(())

        bs = y.size(0)
        total_samples += bs
        sum_nll += float(nll.item()) * bs
        sum_sbece += float(sbece.item()) * bs

        pred = logits.argmax(dim=1)
        correct += pred.eq(y).sum().item()

    denom = max(total_samples, 1)
    return {
        "nll":   sum_nll / denom,
        "sbece": sum_sbece / denom,
        "acc1":  100.0 * correct / denom,
    }


# ============================================================
# 8) Beta warmup INSIDE the 90 epochs
# ============================================================
def get_beta_sbece(epoch, beta_target=1.0, warmup_epochs=5):
    """
    IMPORTANT:
    - total training is still 90 epochs
    - this only ramps beta during the first few epochs
    """
    if warmup_epochs <= 0:
        return beta_target
    if epoch >= warmup_epochs:
        return beta_target
    return beta_target * float(epoch + 1) / warmup_epochs


# ============================================================
# 9) Main 90-epoch training loop
# ============================================================
def run_main_gated_training_90_epochs(
    train_loader,
    val_loader,
    save_dir="PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/SOFT_GATE_90E",
    num_classes=1000,
    epochs=90,
    base_lr=0.1,                 # for batch size ~256
    beta_sbece=1.0,              # consistent with CIFAR
    lambda_sparse=1e-4,
    sbece_bins=15,
    sbece_temp=0.05,
    sbece_every=1,               # can set 2 or 4 if needed
    warmup_beta_epochs=5,        # inside the 90
    grad_clip=5.0,
):
    os.makedirs(save_dir, exist_ok=True)

    # -------------------------
    # Build model
    # -------------------------
    model = GatedResNet50(num_classes=num_classes, init_weights=False)
    model = load_torchvision_pretrained_into_gated(model)
    model = model.to(DEVICE)

    # -------------------------
    # Losses
    # -------------------------
    nll_loss_fn = nn.CrossEntropyLoss()
    sbece_loss_fn = SoftBinnedECELoss(bins=sbece_bins, temperature=sbece_temp).to(DEVICE)

    # -------------------------
    # Optimizer + scheduler
    # -------------------------
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,
        momentum=0.9,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[30, 60],
        gamma=0.1
    )

    # -------------------------
    # Logging
    # -------------------------
    history = []
    best_acc = -1.0
    best_state = None

    print("=" * 80)
    print("Starting MAIN gated training on ImageNet")
    print(f"Epochs          : {epochs}")
    print(f"beta_sbece      : {beta_sbece}")
    print(f"lambda_sparse   : {lambda_sparse}")
    print(f"sbece_every     : {sbece_every}")
    print(f"warmup_beta_ep  : {warmup_beta_epochs}")
    print("=" * 80)

    # -------------------------
    # Epoch loop
    # -------------------------
    for epoch in range(epochs):
        t0 = time.time()

        beta_eff = get_beta_sbece(
            epoch=epoch,
            beta_target=beta_sbece,
            warmup_epochs=warmup_beta_epochs
        )

        train_stats = train_one_epoch_nll_sbece(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=DEVICE,
            nll_loss_fn=nll_loss_fn,
            sbece_loss_fn=sbece_loss_fn,
            beta_sbece=beta_eff,
            lambda_sparse=lambda_sparse,
            sbece_every=sbece_every,
            grad_clip=grad_clip,
        )

        val_stats = evaluate_one_epoch(
            model=model,
            loader=val_loader,
            device=DEVICE,
            nll_loss_fn=nll_loss_fn,
            sbece_loss_fn=sbece_loss_fn,
        )

        scheduler.step()

        # Gate summary
        gates = gather_all_gates(model)
        gate_mean = float(gates.mean().item()) if gates.numel() > 0 else 1.0
        gate_min  = float(gates.min().item()) if gates.numel() > 0 else 1.0
        gate_max  = float(gates.max().item()) if gates.numel() > 0 else 1.0

        epoch_time = time.time() - t0
        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch + 1,
            "lr": current_lr,
            "beta_sbece_eff": beta_eff,

            "train_loss": train_stats["loss"],
            "train_nll": train_stats["nll"],
            "train_sbece": train_stats["sbece"],
            "train_sparse": train_stats["sparse"],
            "train_acc1": train_stats["acc1"],

            "val_nll": val_stats["nll"],
            "val_sbece": val_stats["sbece"],
            "val_acc1": val_stats["acc1"],

            "gate_mean": gate_mean,
            "gate_min": gate_min,
            "gate_max": gate_max,

            "epoch_time_sec": epoch_time,
        }
        history.append(row)

        # Save best by val acc
        if val_stats["acc1"] > best_acc:
            best_acc = val_stats["acc1"]
            best_state = copy.deepcopy(model.state_dict())

            torch.save(
                {
                    "epoch": epoch + 1,
                    "state_dict": model.state_dict(),
                    "best_val_acc1": best_acc,
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                },
                os.path.join(save_dir, "best_gated_model.pth")
            )

        # Save latest every epoch
        torch.save(
            {
                "epoch": epoch + 1,
                "state_dict": model.state_dict(),
                "best_val_acc1": best_acc,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
            },
            os.path.join(save_dir, "latest_gated_model.pth")
        )

        # CSV log
        df_hist = pd.DataFrame(history)
        df_hist.to_csv(os.path.join(save_dir, "training_log.csv"), index=False)

        print(
            f"[Epoch {epoch+1:03d}/{epochs}] "
            f"lr={current_lr:.5f} | "
            f"beta={beta_eff:.3f} | "
            f"train_acc={train_stats['acc1']:.2f} | "
            f"val_acc={val_stats['acc1']:.2f} | "
            f"train_nll={train_stats['nll']:.4f} | "
            f"val_nll={val_stats['nll']:.4f} | "
            f"train_sbece={train_stats['sbece']:.4f} | "
            f"val_sbece={val_stats['sbece']:.4f} | "
            f"gate_mean={gate_mean:.4f} | "
            f"time={epoch_time/60:.1f}m"
        )

    # Save final
    torch.save(
        {
            "epoch": epochs,
            "state_dict": model.state_dict(),
            "best_val_acc1": best_acc,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
        },
        os.path.join(save_dir, "final_gated_model.pth")
    )

    # Restore best in memory (optional)
    if best_state is not None:
        model.load_state_dict(best_state)

    df_hist = pd.DataFrame(history)
    df_hist.to_csv(os.path.join(save_dir, "training_log.csv"), index=False)

    print("=" * 80)
    print(f"Training finished. Best val acc = {best_acc:.2f}")
    print(f"Saved in: {save_dir}")
    print("=" * 80)

    return model, df_hist

In [15]:
from torchvision import datasets, transforms

# ----------- DataLoader -----------
data_dir = "data/imagenet"
batch_size = 16
num_workers = 8
images_dir = 'data/imagenet/val'
labels_file = 'data/imagenet/devkit/ILSVRC2012_devkit_t12/data/val_filenames.txt'
val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
])
val_dataset = ImageNetValDataset(images_dir, labels_file, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
train_dir = os.path.join(data_dir, "train")
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    
test_loader = val_loader

In [ ]:
model_90e, df_log_90e = run_main_gated_training_90_epochs(
    train_loader=train_loader,
    val_loader=val_loader,
    save_dir="PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/SOFT_GATE_90E",
    num_classes=1000,
    epochs=90,
    base_lr=0.05,
    beta_sbece=1.0,
    lambda_sparse=1e-4,
    sbece_bins=15,
    sbece_temp=0.05,
    sbece_every=1,
    warmup_beta_epochs=5,
    grad_clip=5.0,
)

Loaded pretrained weights into gated model. Copied 320 tensors.
Starting MAIN gated training on ImageNet
Epochs          : 90
beta_sbece      : 1.0
lambda_sparse   : 0.0001
sbece_every     : 1
warmup_beta_ep  : 5


In [ ]:
@torch.no_grad()
def harden_gates_per_layer(
    model: nn.Module,
    keep_ratio: float,
    keep_logit: float = 10.0,
    prune_logit: float = -10.0,
):
    """
    Per-layer hardening:
    keep_ratio = fraction of channels kept PER gate layer

    Example:
        keep_ratio = 0.9  -> keep top 90% channels in each gate layer
        keep_ratio = 0.5  -> keep top 50% channels in each gate layer

    Result:
        kept channels  -> gate logits = +10  (gate ~ 1)
        pruned channels -> gate logits = -10 (gate ~ 0)
    """
    for m in model.modules():
        if isinstance(m, ChannelGate):
            g = m.gate_values()  # [C]
            C = g.numel()

            # number of channels to keep in THIS gate layer
            k = int(round(keep_ratio * C))
            k = max(1, min(k, C))   # keep at least 1, at most all

            # top-k threshold
            thresh = torch.topk(g, k, largest=True).values.min()
            keep = g >= thresh

            m.logits.copy_(
                torch.where(
                    keep,
                    torch.full_like(m.logits, keep_logit),
                    torch.full_like(m.logits, prune_logit),
                )
            )

In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def reset_bn_running_stats(model: nn.Module):
    """
    Reset BatchNorm running statistics.
    """
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.running_mean.zero_()
            m.running_var.fill_(1.0)
            if hasattr(m, "num_batches_tracked"):
                m.num_batches_tracked.zero_()


@torch.no_grad()
def update_bn_stats(model: nn.Module, loader, device, num_batches=100):
    """
    Recompute BN running stats using a few forward passes.
    Only BN layers are put in train mode; everything else stays in eval mode.
    """
    # 1) Put whole model in eval first
    model.eval()

    # 2) Then set only BN layers to train
    bn_layers = []
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train()
            bn_layers.append(m)

    n = 0
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        _ = model(x)

        n += 1
        if n >= num_batches:
            break

    # 3) Return model to eval mode after recalibration
    model.eval()

    print(f"BN stats updated using {n} batches.")

In [ ]:
update_bn_stats(model_p, bn_loader, DEVICE, num_batches=100)

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn


def set_gate_trainable(model: nn.Module, trainable: bool):
    for m in model.modules():
        if isinstance(m, ChannelGate):
            m.logits.requires_grad_(trainable)


def load_checkpoint_strict(model: nn.Module, ckpt_path: str, device):
    ckpt = torch.load(ckpt_path, map_location=device)

    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt:
        sd = ckpt["model"]
    else:
        sd = ckpt

    missing, unexpected = model.load_state_dict(sd, strict=True)
    print(f"[load] {ckpt_path}")
    print(f"       missing={len(missing)} unexpected={len(unexpected)}")

    if len(missing) > 0:
        print("       missing (first 10):", missing[:10])
    if len(unexpected) > 0:
        print("       unexpected (first 10):", unexpected[:10])

    model.to(device)
    return model


@torch.no_grad()
def gate_stats(model: nn.Module):
    vals = []
    for m in model.modules():
        if isinstance(m, ChannelGate):
            vals.append(m.gate_values().detach().flatten())

    if len(vals) == 0:
        return 0.0, 0.0

    g = torch.cat(vals, dim=0)
    return float(g.mean().item()), float((g > 0.5).float().mean().item())


@torch.no_grad()
def stage_and_global_keep(model: nn.Module):
    stage_vals = {"layer1": [], "layer2": [], "layer3": [], "layer4": []}
    all_masks = []

    for name, m in model.named_modules():
        if isinstance(m, ChannelGate):
            g = m.gate_values().detach()
            mask = (g > 0.5).float()
            all_masks.append(mask)

            for s in ["layer1", "layer2", "layer3", "layer4"]:
                if name.startswith(s):
                    stage_vals[s].append(mask.mean().item())
                    break

    if len(all_masks) == 0:
        keep_global = 1.0
    else:
        keep_global = torch.cat(all_masks).mean().item()

    keep_stage = {
        s: float(np.mean(v)) if len(v) > 0 else 1.0
        for s, v in stage_vals.items()
    }

    return keep_global, keep_stage


@torch.no_grad()
def effective_flop_reduction_proxy(model: nn.Module):
    keep_global, keep_stage = stage_and_global_keep(model)
    eff_flop_reduction = 100.0 * (1.0 - (keep_global ** 2))

    return {
        "keep_global": keep_global,
        "effective_flop_reduction_%": eff_flop_reduction,
        "keep_layer1": keep_stage["layer1"],
        "keep_layer2": keep_stage["layer2"],
        "keep_layer3": keep_stage["layer3"],
        "keep_layer4": keep_stage["layer4"],
    }


def create_pruned_model(
    ckpt_path,
    keep_ratio,
    bn_loader,
    val_loader=None,
    bn_num_batches=100,
    device=DEVICE,
):
    """
    Load trained gated checkpoint -> harden -> freeze -> BN recalibrate
    Returns the pruned model, ready for recovery fine-tuning.
    """
    model_p = GatedResNet50(num_classes=1000).to(device)
    model_p = load_checkpoint_strict(model_p, ckpt_path, device)

    # prune by hardening
    harden_gates_per_layer(model_p, keep_ratio=keep_ratio)

    # freeze binary gates
    set_gate_trainable(model_p, False)

    # BN recalibration
    reset_bn_running_stats(model_p)
    update_bn_stats(model_p, bn_loader, device, num_batches=bn_num_batches)

    # diagnostics
    gm, gd = gate_stats(model_p)
    flop_info = effective_flop_reduction_proxy(model_p)

    print(f"[pruned keep={keep_ratio:.2f}] gate_mean={gm:.4f} density@0.5={gd:.4f}")
    print(f"[pruned keep={keep_ratio:.2f}] eff_flop_red={flop_info['effective_flop_reduction_%']:.2f}%")

    metrics = None
    if val_loader is not None:
        model_p.eval()
        metrics = evaluate_all_metrics_imagenet(model_p, val_loader, device)
        print(
            f"[post-prune] acc={metrics['acc']:.2f} | "
            f"nll={metrics['nll']:.4f} | "
            f"ece={metrics['ece']:.4f}"
        )

    return model_p, flop_info, metrics

In [ ]:
import os
import pandas as pd
import numpy as np


def run_pruned_model_sweep(
    ckpt_path,
    out_dir,
    bn_loader,
    val_loader,
    prune_ratios=(0.1, 0.3, 0.5, 0.7, 0.9),
    bn_num_batches=100,
):
    """
    Sweep pruned models only:
      checkpoint -> prune (harden) -> freeze -> BN recalib -> evaluate

    Saves:
      - one checkpoint per pruned model
      - one CSV with post-prune metrics
    """
    os.makedirs(out_dir, exist_ok=True)

    rows = []

    for prune_ratio in prune_ratios:
        keep_ratio = 1.0 - prune_ratio
        tag = f"prune{int(prune_ratio*100):02d}_keep{int(keep_ratio*100):02d}"

        print("\n" + "=" * 90)
        print(f"[{tag}] PRUNED MODEL SWEEP")
        print("=" * 90)

        # create pruned model
        model_p, flop_info, post_metrics = create_pruned_model(
            ckpt_path=ckpt_path,
            keep_ratio=keep_ratio,
            bn_loader=bn_loader,
            val_loader=val_loader,
            bn_num_batches=bn_num_batches,
            device=DEVICE,
        )

        # save pruned checkpoint
        pruned_ckpt_path = os.path.join(out_dir, f"pruned_{tag}.pth")
        torch.save(
            {
                "state_dict": model_p.state_dict(),
                "prune_ratio": prune_ratio,
                "keep_ratio": keep_ratio,
                "post_prune_metrics": post_metrics,
                "flop_info": flop_info,
            },
            pruned_ckpt_path,
        )

        # row
        row = {
            "prune_ratio": prune_ratio,
            "keep_ratio": keep_ratio,

            "post_prune_acc": post_metrics["acc"] if post_metrics is not None else np.nan,
            "post_prune_nll": post_metrics["nll"] if post_metrics is not None else np.nan,
            "post_prune_ece": post_metrics["ece"] if post_metrics is not None else np.nan,
            "post_prune_ts_ece": post_metrics.get("ts_ece", np.nan) if post_metrics is not None else np.nan,

            "keep_global": flop_info["keep_global"],
            "effective_flop_reduction_%": flop_info["effective_flop_reduction_%"],
            "keep_layer1": flop_info["keep_layer1"],
            "keep_layer2": flop_info["keep_layer2"],
            "keep_layer3": flop_info["keep_layer3"],
            "keep_layer4": flop_info["keep_layer4"],

            "pruned_ckpt_path": pruned_ckpt_path,
        }

        rows.append(row)

        # incremental save
        csv_path = os.path.join(out_dir, "pruned_model_sweep.csv")
        pd.DataFrame(rows).to_csv(csv_path, index=False)

        print(f"Saved pruned checkpoint -> {pruned_ckpt_path}")
        print(f"Updated CSV -> {csv_path}")

    print(f"\nDONE -> {csv_path}")
    return csv_path

In [ ]:
csv_pruned = run_pruned_model_sweep(
    ckpt_path="PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/SOFT_GATE_90E/best_gated_model.pth",
    out_dir="PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/PRUNED_SWEEP_90E",
    bn_loader=train_loader,   
    val_loader=val_loader,
    prune_ratios=(0.1, 0.3, 0.5, 0.7, 0.9),   
    bn_num_batches=100,
)

In [ ]:
import os
import time
import pandas as pd
import torch
import torch.nn as nn


def fine_tune_pruned_model(
    model,
    train_loader,
    val_loader,
    device,
    out_dir,
    tag,
    epochs=30,
    base_lr=0.01,
    weight_decay=1e-4,
    beta_sbece=1.0,
    sbece_bins=15,
    sbece_temp=0.05,
    sbece_every=1,
    eval_every=1,
    grad_clip=5.0,
):
    """
    Fine-tune a PRUNED gated model (binary gates already hardened + frozen).
    """

    os.makedirs(out_dir, exist_ok=True)

    # only parameters that still require grad (gates are frozen)
    optimizer = torch.optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=base_lr,
        momentum=0.9,
        weight_decay=weight_decay,
        nesterov=False,
    )

    # 30-epoch recovery schedule
    # LR: 0.01 -> 0.001 at ep 10 -> 0.0001 at ep 20
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[10, 20],
        gamma=0.1,
    )

    nll_loss_fn = nn.CrossEntropyLoss()
    sbece_loss_fn = SoftBinnedECELoss(
        bins=sbece_bins,
        temperature=sbece_temp
    ).to(device)

    best_acc = -1.0
    best_state = None
    rows = []

    t0 = time.time()

    for ep in range(1, epochs + 1):
        # -------------------------
        # Train one epoch
        # -------------------------
        train_stats = train_one_epoch_nll_sbece(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            nll_loss_fn=nll_loss_fn,
            sbece_loss_fn=sbece_loss_fn,
            beta_sbece=beta_sbece,
            lambda_sparse=0.0,   # IMPORTANT: no sparsity during recovery
            sbece_every=sbece_every,
            grad_clip=grad_clip,
        )

        row = {
            "epoch": ep,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "train_loss": float(train_stats["loss"]),
            "train_nll": float(train_stats["nll"]),
            "train_sbece": float(train_stats["sbece"]),
            "train_sparse": float(train_stats.get("sparse", 0.0)),
        }

        # -------------------------
        # Validation
        # -------------------------
        do_eval = (ep % eval_every == 0) or (ep == epochs)

        if do_eval:
            val_stats = evaluate_one_epoch(
                model=model,
                loader=val_loader,
                device=device,
                nll_loss_fn=nll_loss_fn,
                sbece_loss_fn=sbece_loss_fn,
            )

            row.update({
                "val_acc1": float(val_stats["acc1"]),
                "val_nll": float(val_stats["nll"]),
                "val_sbece": float(val_stats["sbece"]),
            })

            if row["val_acc1"] > best_acc:
                best_acc = row["val_acc1"]
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

            print(
                f"   [{tag}] ep {ep:02d}/{epochs} | "
                f"train_loss={row['train_loss']:.4f} "
                f"(nll={row['train_nll']:.4f}, sbece={row['train_sbece']:.4f}) | "
                f"val_acc1={row['val_acc1']:.2f} best={best_acc:.2f}"
            )

        rows.append(row)

        # step scheduler AFTER epoch
        scheduler.step()

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    # -------------------------
    # Final full evaluation
    # -------------------------
    final_metrics = evaluate_all_metrics_imagenet(model, val_loader, device)

    curves_path = os.path.join(out_dir, f"curves_{tag}.csv")
    pd.DataFrame(rows).to_csv(curves_path, index=False)

    best_path = os.path.join(out_dir, f"best_{tag}.pth")
    torch.save(
        {
            "state_dict": model.state_dict(),
            "final_metrics": final_metrics,
            "tag": tag,
        },
        best_path,
    )

    print(f"   [{tag}] Saved curves -> {curves_path}")
    print(f"   [{tag}] Saved best   -> {best_path}")
    print(f"   [{tag}] Time: {(time.time() - t0)/60:.1f} min")

    return final_metrics, curves_path, best_path

In [ ]:
import os
import time
import pandas as pd
import torch
import torch.nn as nn


def fine_tune_pruned_model(
    model,
    train_loader,
    val_loader,
    device,
    out_dir,
    tag,
    epochs=20,
    base_lr=0.01,
    weight_decay=1e-4,
    beta_sbece=1.0,
    sbece_bins=15,
    sbece_temp=0.01,
    sbece_every=1,
    eval_every=1,
    grad_clip=5.0,
):
    """
    Fine-tune a PRUNED gated model (binary gates already hardened + frozen).
    """

    os.makedirs(out_dir, exist_ok=True)

    # only parameters that still require grad (gates are frozen)
    optimizer = torch.optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=base_lr,
        momentum=0.9,
        weight_decay=weight_decay,
        nesterov=False,
    )

    # 30-epoch recovery schedule
    # LR: 0.01 -> 0.001 at ep 10 -> 0.0001 at ep 20
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[10, 20],
        gamma=0.1,
    )

    nll_loss_fn = nn.CrossEntropyLoss()
    sbece_loss_fn = SoftBinnedECELoss(
        bins=sbece_bins,
        temperature=sbece_temp
    ).to(device)

    best_acc = -1.0
    best_state = None
    rows = []

    t0 = time.time()

    for ep in range(1, epochs + 1):
        # -------------------------
        # Train one epoch
        # -------------------------
        train_stats = train_one_epoch_nll_sbece(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            nll_loss_fn=nll_loss_fn,
            sbece_loss_fn=sbece_loss_fn,
            beta_sbece=beta_sbece,
            lambda_sparse=0.0,   # IMPORTANT: no sparsity during recovery
            sbece_every=sbece_every,
            grad_clip=grad_clip,
        )

        row = {
            "epoch": ep,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "train_loss": float(train_stats["loss"]),
            "train_nll": float(train_stats["nll"]),
            "train_sbece": float(train_stats["sbece"]),
            "train_sparse": float(train_stats.get("sparse", 0.0)),
        }

        # -------------------------
        # Validation
        # -------------------------
        do_eval = (ep % eval_every == 0) or (ep == epochs)

        if do_eval:
            val_stats = evaluate_one_epoch(
                model=model,
                loader=val_loader,
                device=device,
                nll_loss_fn=nll_loss_fn,
                sbece_loss_fn=sbece_loss_fn,
            )

            row.update({
                "val_acc1": float(val_stats["acc1"]),
                "val_nll": float(val_stats["nll"]),
                "val_sbece": float(val_stats["sbece"]),
            })

            if row["val_acc1"] > best_acc:
                best_acc = row["val_acc1"]
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

            print(
                f"   [{tag}] ep {ep:02d}/{epochs} | "
                f"train_loss={row['train_loss']:.4f} "
                f"(nll={row['train_nll']:.4f}, sbece={row['train_sbece']:.4f}) | "
                f"val_acc1={row['val_acc1']:.2f} best={best_acc:.2f}"
            )

        rows.append(row)

        # step scheduler AFTER epoch
        scheduler.step()

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state, strict=True)

    # -------------------------
    # Final full evaluation
    # -------------------------
    final_metrics = evaluate_all_metrics_imagenet(model, val_loader, device)

    curves_path = os.path.join(out_dir, f"curves_{tag}.csv")
    pd.DataFrame(rows).to_csv(curves_path, index=False)

    best_path = os.path.join(out_dir, f"best_{tag}.pth")
    torch.save(
        {
            "state_dict": model.state_dict(),
            "final_metrics": final_metrics,
            "tag": tag,
        },
        best_path,
    )

    print(f"   [{tag}] Saved curves -> {curves_path}")
    print(f"   [{tag}] Saved best   -> {best_path}")
    print(f"   [{tag}] Time: {(time.time() - t0)/60:.1f} min")

    return final_metrics, curves_path, best_path

In [ ]:
pruned_ckpt_path = "PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/PRUNED_SWEEP_90E/pruned_prune10_keep90.pth"

model_ft_90, pruned_meta_90 = load_pruned_checkpoint(pruned_ckpt_path, device=DEVICE)

final_metrics_90, curves_csv_90, best_ckpt_90 = fine_tune_pruned_model(
    model=model_ft_90,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    out_dir="PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/RECOVERY_90E",
    tag="prune10_keep90",
    epochs=20,
    base_lr=0.01,
    weight_decay=1e-4,
    beta_sbece=1.0,
    sbece_bins=15,
    sbece_temp=0.05,
    sbece_every=1,
    eval_every=1,
    grad_clip=5.0,
)

In [ ]:
import torch.nn.functional as F

# --------------------------------------------------
# Core metric computation from logits
# --------------------------------------------------

@torch.no_grad()
def compute_metrics_from_logits(logits, labels, device, n_bins=15):

    probs = F.softmax(logits, dim=1)
    conf, preds = probs.max(dim=1)

    correct = preds.eq(labels)

    # Accuracy
    acc = correct.float().mean().item()

    # NLL
    nll = F.cross_entropy(logits, labels).item()

    # ECE
    bin_boundaries = torch.linspace(0, 1, n_bins + 1, device=device)
    ece = torch.zeros(1, device=device)

    for i in range(n_bins):
        mask = (conf > bin_boundaries[i]) & (conf <= bin_boundaries[i + 1])
        if mask.sum() > 0:
            acc_bin = correct[mask].float().mean()
            conf_bin = conf[mask].mean()
            ece += torch.abs(acc_bin - conf_bin) * mask.float().mean()

    ece = ece.item()

    # KS (simple version)
    ks = torch.max(torch.abs(
        torch.sort(conf)[0] -
        torch.linspace(0, 1, len(conf), device=device)
    )).item()

    # Sweep CE (reuse ECE for now if you don’t have separate impl)
    mean_sweep_ce = ece

    # -------------------------
    # Temperature scaling
    # -------------------------
    temperature = torch.nn.Parameter(torch.ones(1, device=device))
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=50)

    def _eval():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits / temperature, labels)
        loss.backward()
        return loss

    optimizer.step(_eval)

    logits_ts = logits / temperature
    probs_ts = F.softmax(logits_ts, dim=1)
    conf_ts, preds_ts = probs_ts.max(dim=1)

    ts_nll = F.cross_entropy(logits_ts, labels).item()

    # TS-ECE
    ece_ts = torch.zeros(1, device=device)
    correct_ts = preds_ts.eq(labels)

    for i in range(n_bins):
        mask = (conf_ts > bin_boundaries[i]) & (conf_ts <= bin_boundaries[i + 1])
        if mask.sum() > 0:
            acc_bin = correct_ts[mask].float().mean()
            conf_bin = conf_ts[mask].mean()
            ece_ts += torch.abs(acc_bin - conf_bin) * mask.float().mean()

    ece_ts = ece_ts.item()

    return {
        "acc": acc,
        "nll": nll,
        "ece": ece,
        "ks": ks,
        "mean_sweep_ce": mean_sweep_ce,
        "ts_ece": ece_ts,
        "ts_nll": ts_nll,
        "ts_temperature": temperature.item(),
    }


In [ ]:
import os
import torch
import pandas as pd
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------------------------------
# Bootstrap utility (ALL metrics via your pipeline)
# --------------------------------------------------

def bootstrap_metrics_from_logits(logits, labels, n_boot=300):
    """
    Bootstrap uncertainty estimation using your
    existing compute_metrics_from_logits().
    Returns std for ALL computed metrics.
    """

    N = logits.shape[0]
    metrics_store = []

    for _ in range(n_boot):
        idx = torch.randint(0, N, (N,), device=logits.device)
        logits_b = logits[idx]
        labels_b = labels[idx]

        metrics_b = compute_metrics_from_logits(
            logits_b,
            labels_b,
            device=DEVICE
        )

        metrics_store.append(metrics_b)

    df_boot = pd.DataFrame(metrics_store)

    std_dict = {}
    for col in df_boot.columns:
        std_dict[col + "_std"] = float(df_boot[col].std())

    return std_dict


# --------------------------------------------------
# Paths
# --------------------------------------------------

BASE_CKPT = "PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/SOFT_GATE_90E/best_gated_model.pth"

RECOVERY_DIR = "PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/RECOVERY_90E"

MODEL_FILES = {
    "BASE_unpruned": BASE_CKPT,
    "keep90_recovered": os.path.join(RECOVERY_DIR, "resnet50_imagenet_recovered_keep90.pth"),
    "keep70_recovered": os.path.join(RECOVERY_DIR, "resnet50_imagenet_recovered_keep70.pth"),
    "keep50_recovered": os.path.join(RECOVERY_DIR, "resnet50_imagenet_recovered_keep50.pth"),
    "keep30_recovered": os.path.join(RECOVERY_DIR, "resnet50_imagenet_recovered_keep30.pth"),
    "keep10_recovered": os.path.join(RECOVERY_DIR, "resnet50_imagenet_recovered_keep10.pth"),
}

# --------------------------------------------------
# Load ImageNet loaders
# --------------------------------------------------

train_loader, val_loader, test_loader = get_imagenet_loaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

# --------------------------------------------------
# Evaluate all models
# --------------------------------------------------

results = []

for name, path in MODEL_FILES.items():

    print("\n====================================")
    print(f"Evaluating: {name}")
    print("====================================")

    ckpt = torch.load(path, map_location=DEVICE)

    model = GatedResNet50(num_classes=1000).to(DEVICE)

    if "state_dict" in ckpt:
        model.load_state_dict(ckpt["state_dict"])
    else:
        model.load_state_dict(ckpt)

    model.eval()

    # --------------------------------------------
    # Collect logits once (IMPORTANT)
    # --------------------------------------------
    logits, labels = collect_logits_and_labels(
        model,
        test_loader,
        DEVICE
    )

    logits = logits.to(DEVICE)
    labels = labels.to(DEVICE)

    # --------------------------------------------
    # Compute metrics directly from logits
    # (avoids double evaluation)
    # --------------------------------------------
    metrics = compute_metrics_from_logits(
        logits,
        labels,
        device=DEVICE
    )

    # --------------------------------------------
    # Bootstrap uncertainty (ALL metrics)
    # --------------------------------------------
    boot = bootstrap_metrics_from_logits(
        logits,
        labels,
        n_boot=300
    )

    print("Metrics:", metrics)
    print("Bootstrap std:", boot)

    row = {"model": name}
    row.update(metrics)
    row.update(boot)

    results.append(row)

# --------------------------------------------------
# Save CSV
# --------------------------------------------------

df = pd.DataFrame(results)

out_csv = os.path.join(
    RECOVERY_DIR,
    "imagenet_all_metrics_with_bootstrap.csv"
)

df.to_csv(out_csv, index=False)

print("\nSaved summary ->", out_csv)

df


In [ ]:
# --------------------------------------------------
# RAW METRICS (mean ± std)
# --------------------------------------------------

df_raw = df.copy()

# Accuracy (%)
df_raw["acc (%)"] = (
    (df_raw["acc"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["acc_std"] * 100).round(2).astype(str)
)

# NLL (raw scale, not %)
df_raw["nll"] = (
    df_raw["nll"].round(3).astype(str)
    + " ± "
    + df_raw["nll_std"].round(3).astype(str)
)

# ECE (%)
df_raw["ece (%)"] = (
    (df_raw["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ece_std"] * 100).round(2).astype(str)
)

# KS (%)
df_raw["ks (%)"] = (
    (df_raw["ks"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["ks_std"] * 100).round(2).astype(str)
)

# Mean Sweep CE (%)
df_raw["mean_sweep_ce (%)"] = (
    (df_raw["mean_sweep_ce"] * 100).round(2).astype(str)
    + " ± "
    + (df_raw["mean_sweep_ce_std"] * 100).round(2).astype(str)
)

# Final column selection
df_raw = df_raw[[
    "model",
    "acc (%)",
    "nll",
    "ece (%)",
    "ks (%)",
    "mean_sweep_ce (%)"
]]

df_raw.to_csv("PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/RECOVERY/resnet50_imagenet_raw_metrics.csv", index=False)

print("\nSaved -> imagenet_raw_metrics_paper.csv")
display(df_raw)


In [ ]:
# --------------------------------------------------
# TEMPERATURE SCALED METRICS (mean ± std)
# --------------------------------------------------

df_ts = df.copy()

# Raw ECE (%)
df_ts["raw_ece (%)"] = (
    (df_ts["ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ece_std"] * 100).round(2).astype(str)
)

# Raw NLL
df_ts["raw_nll"] = (
    df_ts["nll"].round(3).astype(str)
    + " ± "
    + df_ts["nll_std"].round(3).astype(str)
)

# TS-ECE (%)
df_ts["ts_ece (%)"] = (
    (df_ts["ts_ece"] * 100).round(2).astype(str)
    + " ± "
    + (df_ts["ts_ece_std"] * 100).round(2).astype(str)
)

# TS-NLL (raw scale)
df_ts["ts_nll"] = (
    df_ts["ts_nll"].round(3).astype(str)
    + " ± "
    + df_ts["ts_nll_std"].round(3).astype(str)
)

# Temperature (no bootstrap needed usually)
df_ts["temperature"] = df_ts["ts_temperature"].round(3)

df_ts = df_ts[[
    "model",
    "raw_ece (%)",
    "raw_nll",
    "ts_ece (%)",
    "ts_nll",
    "temperature"
]]

df_ts.to_csv("PRUNING/RESNET50_IMAGENET/NLL_SBECE_PRUNE/RECOVERY/imagenet_temperature_scaled_metrics_paper.csv", index=False)

print("\nSaved -> imagenet_temperature_scaled_metrics_paper.csv")
display(df_ts)
